# 01 - Exploratory data analysis

What the demand series actually looks like before we model it: level, weekly and
yearly seasonality, promotions, and how much of the variance is noise.

This notebook is executed on every pull request by `pytest --nbmake`, so it has to
run top to bottom with no manual steps and no network access.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf

from forecasting.data import DATE_COLUMN, TARGET_COLUMN, aggregate_total, load_sales

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 3.5)

sales = load_sales()
sales.head()

## Shape and coverage

In [ ]:
print(f"rows        : {len(sales):,}")
print(f"skus        : {', '.join(sorted(sales['sku'].unique()))}")
print(f"date range  : {sales[DATE_COLUMN].min():%Y-%m-%d} to {sales[DATE_COLUMN].max():%Y-%m-%d}")
print(f"missing     : {int(sales.isna().sum().sum())}")
print(f"duplicates  : {int(sales.duplicated(subset=['sku', DATE_COLUMN]).sum())}")

sales.groupby("sku")[TARGET_COLUMN].describe().round(1)

## The series

Three SKUs at very different scales, so plot them separately rather than on one
shared axis.

In [ ]:
skus = sorted(sales["sku"].unique())
figure, axes = plt.subplots(len(skus), 1, figsize=(11, 2.6 * len(skus)), sharex=True)

for axis, sku in zip(axes, skus, strict=True):
    series = sales.loc[sales["sku"] == sku]
    axis.plot(series[DATE_COLUMN], series[TARGET_COLUMN], linewidth=0.7, color="#4c72b0")
    axis.set_title(sku, loc="left", fontsize=10)
    axis.set_ylabel("units")

axes[-1].set_xlabel("date")
figure.tight_layout()

## Total demand

The default modelling target is the sum across SKUs. Per-SKU forecasting is not
implemented yet.

In [ ]:
total = aggregate_total(sales)

figure, axis = plt.subplots()
axis.plot(total[DATE_COLUMN], total[TARGET_COLUMN], linewidth=0.7, color="#333333")
axis.plot(
    total[DATE_COLUMN],
    total[TARGET_COLUMN].rolling(28, center=True).mean(),
    linewidth=2,
    color="#c44e52",
    label="28-day centred mean",
)
axis.set_title("Total daily units", loc="left")
axis.legend()
figure.tight_layout()

print(f"mean {total[TARGET_COLUMN].mean():.1f}, std {total[TARGET_COLUMN].std():.1f}")

## Weekly seasonality

The strongest signal in the data. Saturday runs well above the weekly mean,
Tuesday well below.

In [ ]:
total_by_weekday = total.assign(weekday=total[DATE_COLUMN].dt.day_name())
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_mean = total_by_weekday.groupby("weekday")[TARGET_COLUMN].mean().reindex(order)

figure, axis = plt.subplots(figsize=(8, 3))
axis.bar(weekday_mean.index, weekday_mean.to_numpy(), color="#55a868")
axis.axhline(total[TARGET_COLUMN].mean(), color="#c44e52", linestyle="--", label="overall mean")
axis.set_ylabel("mean units")
axis.set_title("Mean total demand by weekday", loc="left")
axis.tick_params(axis="x", rotation=30)
axis.legend()
figure.tight_layout()

(weekday_mean / total[TARGET_COLUMN].mean()).round(3).to_frame("index_vs_mean")

## Yearly seasonality and trend

In [ ]:
monthly = (
    total.assign(month=total[DATE_COLUMN].dt.month, year=total[DATE_COLUMN].dt.year)
    .groupby(["year", "month"])[TARGET_COLUMN]
    .mean()
    .unstack("year")
)

figure, axis = plt.subplots(figsize=(9, 3.5))
for year in monthly.columns:
    axis.plot(monthly.index, monthly[year], marker="o", markersize=3, label=str(year))
axis.set_xlabel("month")
axis.set_ylabel("mean daily units")
axis.set_title("Mean daily demand by month and year", loc="left")
axis.legend(title="year")
figure.tight_layout()

monthly.round(1)

## Autocorrelation

Spikes at lags 7, 14 and 21 confirm the weekly cycle, which is why the seasonal
naive baseline uses `season_length=7` and SARIMAX uses a seasonal period of 7.

In [ ]:
figure, axis = plt.subplots(figsize=(10, 3.2))
plot_acf(total[TARGET_COLUMN], lags=35, ax=axis, zero=False)
axis.set_title("Autocorrelation of total daily units", loc="left")
figure.tight_layout()

## Promotions

Promotions run in blocks and lift demand materially, so `on_promotion` is a
candidate exogenous regressor. It is known in advance for future dates, unlike
anything derived from `units`.

In [ ]:
promo_effect = (
    sales.groupby(["sku", "on_promotion"])[TARGET_COLUMN]
    .mean()
    .unstack("on_promotion")
    .rename(columns={False: "off_promo", True: "on_promo"})
)
promo_effect["lift"] = promo_effect["on_promo"] / promo_effect["off_promo"] - 1
promo_effect.round(3)

In [ ]:
promo_days = sales.groupby("sku")["on_promotion"].mean()
print("share of days on promotion:")
print((promo_days * 100).round(1).to_string())

## What this means for modelling

- Weekly seasonality dominates, so any model that ignores day-of-week will lose to
  the seasonal naive baseline.
- There is a mild yearly cycle and a small trend, which is what SARIMAX's
  differencing term is for.
- Promotions are a real, known-in-advance driver that no model currently uses.
- Everything derived from `units` has to be lagged. See
  `src/forecasting/features.py`.

Model comparison is in `02-baseline-model.ipynb`.